# Demo 03b — The Perceptron

**Class 03 · Block 2** (slides: `slides/03_linear_models.md`)

One live demo on the **spam** data (Spambase: 4601 e-mails, 57 word/character frequencies, 39% spam):

* **P1 — The perceptron:** the update rule as a `jax.grad`, convergence, XOR, and the same rule as scikit-learn.

The next notebooks are `03_logistic_regression.ipynb` and `04_classification_metrics.ipynb`.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")

In [ ]:
spam = pl.read_csv(DATA / "spambase.csv.zst")
names = [c for c in spam.columns if c != "spam"]
X_raw = spam.select(names).to_numpy().astype(float)
y = spam["spam"].to_numpy().astype(int)
idx_tr, idx_te = (np.asarray(a) for a in train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0))
X_tr_raw, X_te_raw, ytr, yte = X_raw[idx_tr], X_raw[idx_te], y[idx_tr], y[idx_te]
# Most frequencies are 0 and a few are huge: log(1 + x), then standardize with the training statistics.
scaler = StandardScaler().fit(np.log1p(X_tr_raw))
Xtr, Xte = np.asarray(scaler.transform(np.log1p(X_tr_raw))), np.asarray(scaler.transform(np.log1p(X_te_raw)))
print("train", Xtr.shape, "test", Xte.shape, "| spam fraction: train", ytr.mean().round(3), "test", yte.mean().round(3))
print("duplicate e-mails in the data:", int(spam.is_duplicated().sum()), "of", spam.height, "(a leak between train and test)")

### Data visualization

Look before you model. The target is imbalanced (39% spam); most entries are exactly 0; a few features separate the
classes well; the capital-run length spans orders of magnitude, which is why we take $\log(1 + x)$ above.

In [ ]:
order = np.argsort(-np.abs(np.log1p(X_raw[y == 1]).mean(0) - np.log1p(X_raw[y == 0]).mean(0)))[:10]
gap = np.log1p(X_raw[y == 1]).mean(0)[order] - np.log1p(X_raw[y == 0]).mean(0)[order]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
sns.barplot(x=[(y == 0).mean(), (y == 1).mean()], y=["ham", "spam"], ax=axes[0])
axes[0].set(xlabel="fraction of e-mails", title="class balance")
sns.barplot(x=gap, y=[names[j] for j in order], ax=axes[1])
axes[1].set(xlabel="mean log(1 + x): spam minus ham", title="ten most discriminating features")
sns.histplot(x=np.log10(X_raw[:, names.index("capital_run_length_total")]), hue=np.where(y == 1, "spam", "ham"), bins=30, stat="percent", common_norm=False, ax=axes[2])
axes[2].set(xlabel="log10 of capital letters (total)", title="skewed, and longer for spam")
plt.tight_layout()
plt.show()
print("fraction of entries that are exactly 0:", round(float((X_raw == 0).mean()), 3))

## P1 — The perceptron

$\hat y = \text{sign}(w^\top x + b)$, labels $y \in \{-1, +1\}$. The loss is the **perceptron criterion**

$$\ell(w, b) = \max\big(0,\; -y\,(w^\top x + b)\big)$$

zero for a correctly classified point, positive for a mistake. For one example with a mistake,
$\nabla_w \ell = -y\,x$ and $\nabla_b \ell = -y$, so a gradient step of length 1 is
$w \leftarrow w + y\,x,\; b \leftarrow b + y$: **the classic rule is a subgradient step**. We never write it: JAX
differentiates the loss of one example.

In [ ]:
def perceptron_loss(params, x, y_):
    margin = y_ * (x @ params["w"] + params["b"])
    return jnp.mean(jnp.where(margin <= 0, -margin, 0.0))          # a point on the boundary is a mistake


grad_fn = jax.jit(jax.grad(perceptron_loss))


def train_perceptron(X_, y_pm, epochs, record=False):
    params = {"w": jnp.zeros(X_.shape[1]), "b": jnp.zeros(())}
    X_, y_pm = jnp.asarray(X_), jnp.asarray(y_pm, dtype=float)
    mistakes, snapshots = [], []
    for _ in range(epochs):
        wrong = 0
        for i in range(len(y_pm)):
            g = grad_fn(params, X_[i : i + 1], y_pm[i : i + 1])
            wrong += int(g["b"] != 0)                              # non-zero gradient = mistake
            params = jax.tree.map(lambda p, gp: p - gp, params, g)
        mistakes.append(wrong)
        if record:
            snapshots.append((np.asarray(params["w"]).copy(), float(params["b"])))
    return params, mistakes, snapshots


def blobs(rng, n, gap):
    labels = rng.integers(0, 2, n)
    centres = np.array([[-gap / 2, -gap / 2], [gap / 2, gap / 2]])
    return centres[labels] + rng.normal(0, 1.0, (n, 2)), labels


rng = np.random.default_rng(1)
Xb, yb = blobs(rng, 60, 4.0)
_, mist, snaps = train_perceptron(Xb, 2 * yb - 1, 4, record=True)
print("separable blobs, mistakes per epoch:", mist)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True)
xs = np.linspace(-4, 4, 10)
for ax, (w_, b_), title in zip(axes, snaps[:3], ("after epoch 1", "after epoch 2", "after epoch 3")):
    ax.scatter(*Xb[yb == 0].T, s=12, color="tab:blue")
    ax.scatter(*Xb[yb == 1].T, s=12, color="tab:red")
    ax.plot(xs, -(w_[0] * xs + b_) / w_[1], "k")
    ax.set(title=title, ylim=(-4, 4))
plt.show()

**Convergence.** If the classes are linearly separable, the perceptron makes a *finite* number of mistakes and then
stops (Novikoff's theorem: at most $(R/\gamma)^2$ updates, with $R$ the largest $\lVert x\rVert$ and $\gamma$ the margin).
If they overlap, or are not linearly separable at all, it never stops.

In [ ]:
Xo, yo = blobs(rng, 100, 1.5)
_, mist_o, _ = train_perceptron(Xo, 2 * yo - 1, 12)
print("overlapping blobs, mistakes per epoch:", mist_o)

Xx = rng.uniform(0.15, 1.0, (200, 2)) * rng.choice([-1.0, 1.0], (200, 2))            # keep a margin from the axes
yx = (Xx[:, 0] * Xx[:, 1] < 0).astype(int)                       # XOR: opposite signs
p_x, mist_x, _ = train_perceptron(Xx, 2 * yx - 1, 20)
acc_x = np.mean((np.asarray(Xx @ p_x["w"] + p_x["b"]) > 0) == yx)
print(f"XOR, raw features: mistakes per epoch {mist_x[:6]}..., accuracy {acc_x:.2f}")

P = PolynomialFeatures(2, include_bias=False).fit_transform(Xx)   # x1, x2, x1^2, x1 x2, x2^2
p_p, mist_p, _ = train_perceptron(P, 2 * yx - 1, 20)
print(f"XOR, degree-2 features: mistakes per epoch {mist_p}")

The perceptron cannot learn XOR **in the plane** (no line separates it); with the extra column $x_1 x_2$ it can. A
1969 book by Minsky and Papert made this limitation famous. We meet the way out (hidden layers) in Class 05.

**The same rule as scikit-learn.** With `eta0=1`, no penalty and no shuffling, `Perceptron` runs exactly this
online update.

In [ ]:
p_spam, mist_spam, _ = train_perceptron(Xtr, 2 * ytr - 1, 1)
sk_p = Perceptron(penalty=None, eta0=1.0, shuffle=False, tol=None, max_iter=1).fit(Xtr, ytr)  # pyright: ignore[reportArgumentType]
print("max |weights ours - sklearn| =", np.abs(np.asarray(p_spam["w"]) - np.asarray(sk_p.coef_)[0]).max(), "| mistakes in the epoch:", mist_spam[0], "of", len(ytr))
print("test accuracy after one epoch:", round(float(np.mean((np.asarray(Xte @ p_spam["w"] + p_spam["b"]) > 0) == yte)), 3))

## Summary

* The perceptron's update is the subgradient of $\max(0, -m)$; it converges only on separable data.
* XOR defeats a straight line; an extra feature ($x_1 x_2$, or a hidden layer in Class 05) fixes it.
* It has no probability model: it answers yes or no, and it is **not** a generalized linear model.
* Next: logistic regression gives a probability (`03_logistic_regression.ipynb`).